# Grain Size Analysis via SAM 2

---
### Requirements
```bash
pip install torch torchvision
pip install git+https://github.com/facebookresearch/sam2.git
pip install opencv-python numpy pandas matplotlib scipy ipywidgets
```
Download SAM 2 checkpoint from the [official repo](https://github.com/facebookresearch/sam2#model-checkpoints).  


In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
import os
import torch

from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

%matplotlib inline
print("All imports OK")
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : CPU (forced)")


---
## 1 · Inputs
Set image path, scale, and SAM 2 checkpoint below.


In [ ]:
# ── USER INPUTS ─────────────────────────────────────────────────────────────

IMAGE_PATH    = "image.png"          # path to POM image
MM_PER_PIXEL  = 0.000423                  # scale factor (mm / pixel)

SAM2_CHECKPOINT = "sam2.1_hiera_tiny.pt"  # <-- path to downloaded checkpoint
SAM2_CONFIG     = "configs/sam2.1/sam2.1_hiera_tiny.yaml"  # bundled with sam2

# ── PRE-PROCESSING PARAMETERS ────────────────────────────────────────────────
GAUSSIAN_KSIZE = (5, 5)   # kernel size — increase for high-res images
GAUSSIAN_SIGMA = 1.0
MEDIAN_KSIZE   = 5        # must be odd

# ── SEGMENTATION FILTER ──────────────────────────────────────────────────────
MIN_GRAIN_AREA_PX = 200   # discard masks smaller than this (px²)


---
## 2 · Load & Preview Image


In [ ]:
img_bgr = cv2.imread(IMAGE_PATH)
assert img_bgr is not None, f"Could not load image: {IMAGE_PATH}"
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

h, w = img_rgb.shape[:2]
print(f"Image size : {w} × {h} px")
print(f"Physical   : {w * MM_PER_PIXEL:.3f} × {h * MM_PER_PIXEL:.3f} mm")

plt.figure(figsize=(7, 5))
plt.imshow(img_rgb)
plt.title("Raw POM Image")
plt.axis("off")
plt.tight_layout()
plt.show()


---
## 3 · Pre-processing (Gaussian and Median blur)


In [ ]:
def preprocess(image_bgr):
    blurred = cv2.GaussianBlur(image_bgr, GAUSSIAN_KSIZE, GAUSSIAN_SIGMA)
    blurred = cv2.medianBlur(blurred, MEDIAN_KSIZE)
    return blurred

preprocessed_bgr = preprocess(img_bgr)
preprocessed_rgb = cv2.cvtColor(preprocessed_bgr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(img_rgb);          axes[0].set_title("Original");      axes[0].axis("off")
axes[1].imshow(preprocessed_rgb); axes[1].set_title("Pre-processed"); axes[1].axis("off")
plt.tight_layout()
plt.show()


---
## 4 · Load SAM 2 Model


In [ ]:
device = torch.device("cpu")
print(f"Loading SAM 2 on {device} ...")

sam2_model = build_sam2(SAM2_CONFIG, SAM2_CHECKPOINT, device=device)

mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2_model,
    points_per_side=32,           # ↑ for finer grains
    pred_iou_thresh=0.80,
    stability_score_thresh=0.90,
    box_nms_thresh=0.7,
    min_mask_region_area=MIN_GRAIN_AREA_PX,
)

print("SAM 2 loaded ✓")


---
## 5 · Run Segmentation


In [ ]:
print("Running SAM 2 segmentation — please wait ...")
masks = mask_generator.generate(preprocessed_rgb)
masks = sorted(masks, key=lambda m: m["area"], reverse=True)

# Filter by minimum area
masks = [m for m in masks if m["area"] >= MIN_GRAIN_AREA_PX]
print(f"Segments found (after area filter): {len(masks)}")


---
## 6 · Visualise Segmented Grains


In [ ]:
def colorize_masks(image_rgb, masks):
    overlay     = image_rgb.copy().astype(np.float32) / 255.0
    color_layer = np.zeros_like(overlay)
    np.random.seed(42)
    for m in masks:
        binary = m["segmentation"]
        color  = np.random.rand(3)
        for c in range(3):
            color_layer[:, :, c] += binary * color[c]
    blended = np.clip(0.55 * overlay + 0.45 * color_layer, 0, 1)
    return (blended * 255).astype(np.uint8)

overlay = colorize_masks(img_rgb, masks)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_rgb);  axes[0].set_title("Raw Image");              axes[0].axis("off")
axes[1].imshow(overlay);  axes[1].set_title(f"Segmented ({len(masks)} grains)"); axes[1].axis("off")
plt.suptitle("SAM 2 Grain Segmentation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 7 · Extract Grain Statistics
Grain size is computed as the **equivalent circular diameter**: $d = 2\sqrt{A/\pi}$


In [ ]:
records = []

for i, m in enumerate(masks):
    binary   = m["segmentation"].astype(np.uint8)
    area_px  = int(binary.sum())

    equiv_diam_px  = 2.0 * np.sqrt(area_px / np.pi)
    area_mm2       = area_px * (MM_PER_PIXEL ** 2)
    equiv_diam_um  = equiv_diam_px * MM_PER_PIXEL * 1000.0

    cy, cx = ndimage.center_of_mass(binary)

    records.append({
        "grain_id":          i,
        "area_px":           area_px,
        "area_mm2":          round(area_mm2, 6),
        "equiv_diameter_px": round(equiv_diam_px, 2),
        "equiv_diameter_um": round(equiv_diam_um, 2),
        "centroid_x":        round(cx, 1),
        "centroid_y":        round(cy, 1),
    })

df = pd.DataFrame(records)
print(f"Grains extracted: {len(df)}")
df.head(10)


---
## 8 · Summary Statistics


In [ ]:
summary = df["equiv_diameter_um"].describe().rename("Equiv. Diameter (µm)")
print(summary.to_string())
print(f"\nMedian diameter : {df['equiv_diameter_um'].median():.2f} µm")


---
## 9 · Grain Size Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram of equivalent diameter
axes[0].hist(df["equiv_diameter_um"], bins=35, color="#4C8EDA", edgecolor="white")
mean_d   = df["equiv_diameter_um"].mean()
median_d = df["equiv_diameter_um"].median()
axes[0].axvline(mean_d,   color="red",    linestyle="--", label=f"Mean   {mean_d:.1f} µm")
axes[0].axvline(median_d, color="orange", linestyle="--", label=f"Median {median_d:.1f} µm")
axes[0].set_xlabel("Equivalent Diameter (µm)")
axes[0].set_ylabel("Count")
axes[0].set_title("Grain Size Distribution")
axes[0].legend()

# Cumulative distribution
sorted_d = np.sort(df["equiv_diameter_um"].values)
cdf      = np.arange(1, len(sorted_d) + 1) / len(sorted_d)
axes[1].plot(sorted_d, cdf * 100, color="#4C8EDA", linewidth=2)
axes[1].axvline(mean_d,   color="red",    linestyle="--", label=f"Mean   {mean_d:.1f} µm")
axes[1].axvline(median_d, color="orange", linestyle="--", label=f"Median {median_d:.1f} µm")
axes[1].set_xlabel("Equivalent Diameter (µm)")
axes[1].set_ylabel("Cumulative %")
axes[1].set_title("Cumulative Size Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()


---
## 10 · Save Outputs


In [ ]:
base = os.path.splitext(IMAGE_PATH)[0]

# Save CSV
csv_path = base + "_grain_stats.csv"
df.to_csv(csv_path, index=False)
print(f"CSV saved  → {csv_path}")

# Save overlay figure
fig_path = base + "_segmented.png"
cv2.imwrite(fig_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
print(f"Image saved → {fig_path}")

# Save summary figure
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Grain Size Analysis — SAM 2", fontsize=13, fontweight="bold")
axes[0].imshow(img_rgb);  axes[0].set_title("Raw");          axes[0].axis("off")
axes[1].imshow(overlay);  axes[1].set_title("Segmented");    axes[1].axis("off")
axes[2].hist(df["equiv_diameter_um"], bins=30, color="#4C8EDA", edgecolor="white")
axes[2].axvline(mean_d, color="red", linestyle="--", label=f"Mean {mean_d:.1f} µm")
axes[2].set_xlabel("Equiv. Diameter (µm)"); axes[2].set_ylabel("Count")
axes[2].set_title("Distribution"); axes[2].legend()
plt.tight_layout()
summary_fig_path = base + "_results_summary.png"
plt.savefig(summary_fig_path, dpi=150)
print(f"Summary figure saved → {summary_fig_path}")
plt.show()


---
## Notes for Tuning

| Parameter | Effect | Problem |
|---|---|---|
| `points_per_side` ↑ | Denser prompt grid, finds smaller grains | Small grains are missed |
| `pred_iou_thresh` ↑ | Stricter mask quality | Too many noisy masks |
| `stability_score_thresh` ↑ | Stricter stability | Fragmented/unstable masks |
| `MIN_GRAIN_AREA_PX` ↑ | Filters out small fragments | Lots of tiny artefacts in output |
| `GAUSSIAN_KSIZE` larger | More aggressive smoothing | Grain boundaries look noisy |
| `MEDIAN_KSIZE` larger | Preserves edges while removing noise | Salt-and-pepper noise |

